# 实验 1：有预算时，先保留什么

> 状态：verified；真实运行的是确定性打包逻辑，不含模型调用。

数据为人工构造的运维教学片段。使用显式 **UTF-8 字节 tokenizer**：每个字节算一个 token，因而中文常占 3 个 token。这不是 GPT/Claude 的计费 token；换模型时通过 `count` 参数接入对应 tokenizer，并计入聊天模板和工具 schema。

本实验回答三个可验证问题：是否先过滤跨租户文本、是否保护硬约束、是否给输出留空间。它不能测量 Context Rot 或证明模型答案改善。配套正文：[Context Builder](../02-patterns/01-context-builder.md)。

In [1]:
import sys
from pathlib import Path
sys.path.insert(0,str(Path.cwd()))
from context_lab import Item, pack, byte_tokens
items = [
    Item("goal", "查找 ALM-12003 的当前步骤", 10, True),
    Item("constraint", "不得重启设备；只读取日志", 10, True),
    Item("noise", "已完成任务日志；" * 70, 1),
    Item("evidence", "v2 手册：先读取转速日志，再核对供电。", 9),
    Item("history", "此前检查过 v1，现已被 v2 替代。", 2),
    Item("other-tenant", "别的租户内部记录", 100, tenant="beta"),
]
[(x.id,byte_tokens(x.text)) for x in items]

Out[1]: 
[('goal', 32),
 ('constraint', 36),
 ('noise', 1680),
 ('evidence', 54),
 ('history', 43),
 ('other-tenant', 24)]


假定教学窗口 420 byte-token，预留输出 100，输入最多 320。不可变部分先入窗；剩余片段按 `utility / 实际编码长度` 排序。这是可解释的贪心启发式，不保证全局最优，也不自动知道真实效用。

In [2]:
result=pack(items,window=420,reserved_output=100,tenant="alpha")
print(result["text"])
print({k:v for k,v in result.items() if k!="text"})
assert {"goal","constraint","evidence"} <= set(result["selected_ids"])
assert "other-tenant" not in result["selected_ids"]
assert result["input_tokens"] <= 320

[goal] 查找 ALM-12003 的当前步骤

[constraint] 不得重启设备；只读取日志

[evidence] v2 手册：先读取转速日志，再核对供电。

[history] 此前检查过 v1，现已被 v2 替代。
{'selected_ids': ['goal', 'constraint', 'evidence', 'history'], 'dropped': [{'id': 'other-tenant', 'reason': 'permission'}, {'id': 'noise', 'reason': 'budget'}], 'input_tokens': 212, 'input_limit': 320, 'counting': 'injected counter (default: UTF-8 byte tokenizer)'}


对照：按历史顺序装入文本再硬截断。下例不访问跨租户数据，仍可能因为长噪声占前面而丢掉证据。

In [3]:
baseline="\n\n".join(x.text for x in items if x.tenant=="alpha").encode("utf-8")[:320].decode("utf-8",errors="ignore")
comparison={"naive_evidence_present":"v2 手册" in baseline,"packed_evidence_present":"v2 手册" in result["text"],"naive_bytes":byte_tokens(baseline),"packed_bytes":result["input_tokens"]}
comparison

Out[3]: 
{'naive_evidence_present': False,
 'packed_evidence_present': True,
 'naive_bytes': 318,
 'packed_bytes': 212}


In [4]:
try:
    pack(items,window=80,reserved_output=30,tenant="alpha")
except ValueError as error:
    print(type(error).__name__, str(error))
else:
    raise AssertionError("预算不足必须显式失败")

ValueError mandatory information exceeds budget; do not silently truncate


**读结果**：打包保留关键证据的结论来自片段 ID/文本检查；尚未验证模型是否真正使用这些证据。下一步替换计数器、加入自己的任务，冻结同一模型配置比较答案与工具行为。不要把手工 utility 当训练得到的相关性评分。源代码：[context_lab.py](context_lab.py)。